# Delay Target Analysis

Analyse der Zielvariable `arrival_delay` — Verteilung, OTP-Baseline, Vergleich mit Departure Delay und Delay Delta, Ausfälle.

## Setup

In [ ]:
from zh_tram_flow.notebook import *

TRAIN, TEST, lf = setup_analysis("03_analysis_1-target")

# --- Samples für Visualisierungen ---
# Der vollständige Datensatz (~60M Zeilen) bleibt als LazyFrame für alle Aggregationen.
# Für Plots wird ein festes Sample verwendet:
#   - Seed=42 → immer identische Stichprobe → Ergebnisse zwischen Sessions vergleichbar
#   - 100k Zeilen sind statistisch repräsentativ für Verteilungsplots
#   - Für Details oder Scatter ggf. SAMPLE_LARGE verwenden (500k)
SAMPLE_SMALL = lf.collect().sample(n=100_000, seed=42)
SAMPLE_LARGE = lf.collect().sample(n=500_000, seed=42)

# Train (2023+2024) + Test (2025) kombiniert — für jahresübergreifende Analysen
lf_all = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])

%load_ext autoreload
%autoreload 2

## Target Definition

**Primäres Ziel:** `arrival_delay` — Sekunden Verspätung bei der Ankunft an einer Haltestelle (negativ = zu früh).

Eine verspätete Abfahrt kann noch ausgeglichen werden — eine verspätete Ankunft nicht.   
Sie trifft Fahrgäste direkt: verpasste Anschlüsse, geplatzte Termine, Folgeverspätungen.

| Column | Rolle | Beschreibung |
|:---|:---|:---|
| `arrival_delay` | **Primäres Ziel** | Sekunden Verspätung bei Ankunft — was Fahrgäste erleben |
| `departure_delay` | Feature | Sekunden Verspätung bei Abfahrt — Startzustand für den nächsten Abschnitt |
| `delay_delta` | Abgeleitetes Feature | `departure_delay - arrival_delay` — positiv = Verspätung wächst am Halt, negativ = Verspätung wird abgebaut |

**Was wir nicht direkt sehen:** ob eine Verspätung über mehrere Halte vollständig aufgeholt wurde. `delay_delta` liefert das Signal pro Halt, aber keine Trip-Level-Sicht.

### Delay Overview — Per Year

Größenordnungen im Überblick: mittlere Verspätung pro Halt und Jahr, Min/Max. Alle drei Jahre (2023–2025) aus Train + Test kombiniert.

In [ ]:
section_header("Delay Overview — Per Year")

from wgnd.core.theme import mpl_style

stats_year = (
    lf_all
    .with_columns(pl.col("operating_date").dt.year().alias("year"))
    .group_by("year")
    .agg([
        pl.len().alias("n_stops"),
        pl.col("arrival_delay").mean().alias("arr_mean"),
        pl.col("arrival_delay").median().alias("arr_median"),
        pl.col("arrival_delay").min().alias("arr_min"),
        pl.col("arrival_delay").max().alias("arr_max"),
        pl.col("departure_delay").mean().alias("dep_mean"),
        pl.col("departure_delay").median().alias("dep_median"),
        pl.col("delay_delta").mean().alias("delta_mean"),
        pl.col("delay_delta").median().alias("delta_median"),
    ])
    .sort("year")
    .collect()
    .to_pandas()
)

display_stats = stats_year.copy()
display_stats["n_stops"] = display_stats["n_stops"].apply(lambda x: f"{x:,.0f}")
for col in display_stats.columns[2:]:
    display_stats[col] = display_stats[col].apply(lambda x: f"{x:+.1f}s")
display_stats.columns = ["Year", "Stop Count",
                         "Arr Mean", "Arr Median", "Arr Min", "Arr Max",
                         "Dep Mean", "Dep Median", "Δ Mean", "Δ Median"]
show_df(display_stats)

style  = mpl_style()
years  = stats_year["year"].astype(str).tolist()
x      = np.arange(len(years))
width  = 0.25
colors = cfg.palette_n(3)

fig, ax = plt.subplots(figsize=(10, 4))
for i, (col, label, color) in enumerate(zip(
    ["arr_mean", "dep_mean", "delta_mean"],
    ["Arrival Delay", "Departure Delay", "Delay Delta"],
    colors,
)):
    bars = ax.bar(x + i * width, stats_year[col], width, label=label, color=color)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{bar.get_height():+.1f}s", ha="center", va="bottom", fontsize=9)

ax.axhline(0, color=cfg.ANNO_REF, lw=1, linestyle="--")
ax.set_xticks(x + width)
ax.set_xticklabels(years, fontsize=11)
ax.set_ylabel("Sekunden", **style["label"])
ax.set_title("Ø Verspätung pro Halt — nach Jahr", **style["title"])
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
plt.tight_layout()
plt.show()

**Beobachtung:** Alle drei Delay-Größen steigen über die Jahre leicht an — der markanteste Trend ist `delay_delta`: von +4.4s (2023) auf +7.7s (2025), fast eine Verdopplung. Das bedeutet: Trams verlieren pro Halt im Schnitt immer mehr Zeit. Hypothese: Der Fahrplan ist zu knapp kalkuliert und/oder wachsendes Fahrgastaufkommen verlängert die Haltezeiten. → Vertiefen in `03_analysis_3-temporal`.

In [ ]:
# delay_delta ist bereits im Feature-Set (berechnet in 02_preparation)
# is_recovering kann bei Bedarf abgeleitet werden: delay_delta < 0

## Delay Distribution

Grundform aller drei Delay-Spalten: Minimum, Maximum, Mittelwert, Median, Streuung. Basis für alle weiteren Analysen.

In [ ]:
section_header("Delay Distribution")

from wgnd.core.theme import mpl_style

# --- Stats table — alle 3 Spalten (Full Scan) ---
delay_cols = ["arrival_delay", "departure_delay", "delay_delta"]
rows = []
for col in delay_cols:
    r = (
        lf.select([
            pl.col(col).min().alias("min"),
            pl.col(col).mean().alias("mean"),
            pl.col(col).median().alias("median"),
            pl.col(col).std().alias("std"),
            pl.col(col).max().alias("max"),
        ])
        .collect()
        .to_pandas()
        .assign(column=col)
    )
    rows.append(r)

stats = pd.concat(rows).set_index("column").round(1)
show_df(stats)

# --- Histogramme nebeneinander (SAMPLE_SMALL, geclippt) ---
CLIP   = (-300, 600)
style  = mpl_style()
titles = ["Arrival Delay", "Departure Delay", "Delay Delta"]
colors = cfg.palette_n(3)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title, color in zip(axes, delay_cols, titles, colors):
    data       = SAMPLE_SMALL[col].clip(CLIP[0], CLIP[1]).to_numpy()
    mean_val   = float(SAMPLE_SMALL[col].mean())
    median_val = float(SAMPLE_SMALL[col].median())

    ax.hist(data, bins=80, color=color, alpha=0.85, edgecolor="none")
    ax.axvline(mean_val,   color=cfg.ANNO_MEAN,   lw=1.5,               label=f"Ø {mean_val:.0f}s")
    ax.axvline(median_val, color=cfg.ANNO_MEDIAN, lw=1.5, linestyle="--", label=f"Median {median_val:.0f}s")
    ax.set_title(title, **style["title"])
    ax.set_xlabel("Seconds", **style["label"])
    ax.set_ylabel("Count", **style["label"])
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
    ax.legend(fontsize=9)

plt.suptitle("Distribution — Sample 100k · clipped −300 to +600s", fontsize=11, color=cfg.CHART_TITLE, y=1.01)
plt.tight_layout()
plt.show()

**Beobachtung:** Alle drei Verteilungen sind rechtsschief — wenige extreme Verspätungen ziehen den Mittelwert deutlich über den Median. Bei `delay_delta` gilt: `median(A−B) ≠ median(A) − median(B)`, daher erscheint die Diskrepanz größer als erwartet. Auffällig: Frühankünfte bis −200s trotz dominanter Verspätungsrichtung — mögliche Ursache sind Terminus-Effekte (Trams warten am Wendepunkt und starten früh) oder Fahrtbeginn-Logik. → Räumlich prüfen in `03_analysis_4-spatial`.

## Log Transform — Arrival Delay

Vergleich Original vs. Signed-Log-Transformation von `arrival_delay`. Signed Log erhält das Vorzeichen und komprimiert Extremwerte — zeigt ob die Verteilung modellierbarer wird. Dazu: MAE bei Mittelwert- vs. Median-basierter Vorhersage (Robustness-Check).

In [ ]:
section_header("Log Transform — Arrival Delay")

from wgnd.core.theme import mpl_style

arr    = SAMPLE_SMALL["arrival_delay"].to_numpy()
# Signed Log: Vorzeichen bleibt erhalten, Magnitude wird komprimiert
# sign(x) · log(|x| + 1) → symmetrisch um 0, Extremwerte gezähmt
arr_log = np.sign(arr) * np.log1p(np.abs(arr))

style = mpl_style()
colors = cfg.palette_n(2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# --- Original ---
ax1.hist(np.clip(arr, -300, 600), bins=80, color=colors[0], alpha=0.85, edgecolor="none")
ax1.axvline(np.mean(arr),   color=cfg.ANNO_MEAN,   lw=1.5,                  label=f"Ø {np.mean(arr):.0f}s")
ax1.axvline(np.median(arr), color=cfg.ANNO_MEDIAN, lw=1.5, linestyle="--",  label=f"Median {np.median(arr):.0f}s")
ax1.set_title("Arrival Delay — Original (clipped)", **style["title"])
ax1.set_xlabel("Seconds", **style["label"])
ax1.set_ylabel("Count", **style["label"])
ax1.legend(fontsize=9)
ax1.spines[["top", "right"]].set_visible(False)
ax1.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)

# --- Log-transformiert ---
ax2.hist(arr_log, bins=80, color=colors[1], alpha=0.85, edgecolor="none")
ax2.axvline(np.mean(arr_log),   color=cfg.ANNO_MEAN,   lw=1.5,                  label=f"Ø {np.mean(arr_log):.2f}")
ax2.axvline(np.median(arr_log), color=cfg.ANNO_MEDIAN, lw=1.5, linestyle="--",  label=f"Median {np.median(arr_log):.2f}")
ax2.set_title("Arrival Delay — Signed Log Transform", **style["title"])
ax2.set_xlabel("sign(x) · log(|x| + 1)", **style["label"])
ax2.legend(fontsize=9)
ax2.spines[["top", "right"]].set_visible(False)
ax2.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)

plt.suptitle("Log-Transform komprimiert Extremwerte — Verteilung nähert sich Normalform",
             fontsize=11, color=cfg.CHART_TITLE, y=1.01)
plt.tight_layout()
plt.show()

# --- MAE mit Mittelwert vs. Median als naive Baseline ---
mean_pred   = np.mean(arr)
median_pred = np.median(arr)
mae_mean    = np.mean(np.abs(arr - mean_pred))
mae_median  = np.mean(np.abs(arr - median_pred))

log(f"Naive Baseline — Vorhersage = Mittelwert:  MAE = {mae_mean:.1f}s")
log(f"Naive Baseline — Vorhersage = Median:      MAE = {mae_median:.1f}s  (robuster gegenüber Ausreißern)")
log(f"Differenz: {mae_mean - mae_median:+.1f}s  →  Median reduziert MAE um {(1 - mae_median/mae_mean)*100:.1f}%")

## On-Time Performance (OTP)

Anteil der Halte innerhalb ±120 Sekunden Planzeit — Branchen-Standard KPI für den öffentlichen Nahverkehr. Für `delay_delta`: Anteil der Halte an denen Verspätung abgebaut, neutral oder aufgebaut wird.

In [ ]:
section_header("On-Time Performance")

from wgnd.core.theme import mpl_style

otp = lf.select([
    (pl.col("arrival_delay").abs()   <= 120).mean().alias("arr_on_time"),
    (pl.col("arrival_delay")          > 120).mean().alias("arr_late"),
    (pl.col("arrival_delay")          < -120).mean().alias("arr_early"),
    (pl.col("departure_delay").abs() <= 120).mean().alias("dep_on_time"),
    (pl.col("departure_delay")        > 120).mean().alias("dep_late"),
    (pl.col("departure_delay")        < -120).mean().alias("dep_early"),
    (pl.col("delay_delta")            <  0).mean().alias("delta_recovering"),
    (pl.col("delay_delta")           == 0).mean().alias("delta_neutral"),
    (pl.col("delay_delta")            >  0).mean().alias("delta_growing"),
]).collect()

style      = mpl_style()
x          = np.arange(3)
width      = 0.35
categories = ["On-Time\n(|delay| ≤ 120s)", "Late\n(> 120s)", "Early\n(< −120s)"]
arr_vals   = [otp["arr_on_time"][0], otp["arr_late"][0], otp["arr_early"][0]]
dep_vals   = [otp["dep_on_time"][0], otp["dep_late"][0], otp["dep_early"][0]]
two_colors = cfg.palette_n(2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# --- Arrival vs Departure OTP ---
b1 = ax1.bar(x - width/2, arr_vals, width, label="Arrival",   color=two_colors[0])
b2 = ax1.bar(x + width/2, dep_vals, width, label="Departure", color=two_colors[1])
ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=10)
ax1.set_ylabel("Anteil", **style["label"])
ax1.set_title("Arrival vs Departure — OTP", **style["title"])
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax1.legend()
ax1.spines[["top", "right"]].set_visible(False)
ax1.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
for bar in [*b1, *b2]:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
             f"{bar.get_height():.1%}", ha="center", va="bottom", fontsize=9)

# --- Delay Delta — Recovering vs Growing ---
delta_vals   = [otp["delta_recovering"][0], otp["delta_neutral"][0], otp["delta_growing"][0]]
delta_labels = ["Recovering\n(Δ < 0)", "Neutral\n(Δ = 0)", "Growing\n(Δ > 0)"]
delta_colors = [cfg.COLOR_POSITIVE, cfg.COLOR_NEUTRAL, cfg.COLOR_NEGATIVE]
b3 = ax2.bar(range(3), delta_vals, width=0.5, color=delta_colors)
ax2.set_xticks(range(3))
ax2.set_xticklabels(delta_labels, fontsize=10)
ax2.set_title("Delay Delta — Recovering vs Growing", **style["title"])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax2.spines[["top", "right"]].set_visible(False)
ax2.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
for bar in b3:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
             f"{bar.get_height():.1%}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

**Beobachtung:** 85–87% Pünktlichkeit ist ein guter Wert für ein urbanes Tramnetz (Zürich liegt im europäischen Spitzenfeld). Die Nicht-Pünktlichen sind fast ausschließlich verspätet (13%) — kaum zu früh (0.1%). Das System hat eine klare Bias Richtung Verspätung. Bei `delay_delta`: nur ~27% der Halte zeigen Recovery, ~70% bauen Verspätung auf. Das Netz hat systemisch zu wenig Puffer — Hypothese: Fahrplan zu knapp kalkuliert. Kaskadenwirkungen (eine verspätete Fahrt verzögert die nächste) sind mit `trip_id` jetzt analysierbar — Hotspot-Analyse in `03_analysis_2-network.ipynb`.

## Arrival vs Departure Delay

Alle drei Delay-Spalten nebeneinander als Boxplot — zeigt Lage, Streuung und Ausreißer auf einen Blick. Bauen Halte im Durchschnitt Verspätung auf oder ab?

In [ ]:
section_header("Arrival vs Departure Delay")

from wgnd.core.theme import mpl_style

style  = mpl_style()
colors = cfg.palette_n(3)
labels = ["Arrival\nDelay", "Departure\nDelay", "Delay\nDelta"]

df_box = SAMPLE_SMALL[["arrival_delay", "departure_delay", "delay_delta"]].to_pandas().clip(-300, 600)

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [df_box[col] for col in ["arrival_delay", "departure_delay", "delay_delta"]],
    labels=labels,
    patch_artist=True,
    medianprops=dict(color=cfg.ANNO_MEDIAN, linewidth=2),
    whiskerprops=dict(color=cfg.CHART_AXIS),
    capprops=dict(color=cfg.CHART_AXIS),
    flierprops=dict(marker=".", markersize=1, alpha=0.15, color=cfg.COLOR_NEUTRAL),
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.axhline(0, color=cfg.ANNO_REF, lw=1.2, linestyle="--", label="0s (on time)")
ax.set_ylabel("Seconds", **style["label"])
ax.set_title("Distribution — Sample 100k · clipped −300 to +600s", **style["title"])
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

means = lf.select([
    pl.col("arrival_delay").mean().alias("arr_mean"),
    pl.col("departure_delay").mean().alias("dep_mean"),
    pl.col("delay_delta").mean().alias("delta_mean"),
]).collect()
log(f"Ø Arrival Delay:   {means['arr_mean'][0]:+.1f}s")
log(f"Ø Departure Delay: {means['dep_mean'][0]:+.1f}s")
log(f"Ø Delay Delta:     {means['delta_mean'][0]:+.1f}s  (positiv = Verspätung wächst am Halt)")

**Beobachtung:** `departure_delay` liegt konsistent über `arrival_delay` — Halte kosten Zeit. `delay_delta` ist zentriert nahe 0 mit breiter Streuung: die meisten Halte sind annähernd neutral, aber extreme Werte in beide Richtungen sind vorhanden. Die starke linke Flanke des Delta (starke Recovery) deutet auf wenige Halte mit großem Zeitgewinn hin — wahrscheinlich Endhalte oder Expresssegmente wo Trams Puffer aufholen.

## Delay Delta — Distribution Detail

Separate Betrachtung der `delay_delta` Verteilung im engen Bereich (±100s). Wie ist die Form — symmetrisch, bimodal, stark schief?

In [ ]:
section_header("Delay Delta — Distribution Detail")

from wgnd.core.theme import mpl_style

style = mpl_style()
data_delta  = SAMPLE_SMALL["delay_delta"].clip(-100, 100).to_numpy()
mean_delta   = float(SAMPLE_SMALL["delay_delta"].mean())
median_delta = float(SAMPLE_SMALL["delay_delta"].median())

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(data_delta, bins=120, color=cfg.PALETTE_CATEGORICAL[4], alpha=0.85, edgecolor="none")
ax.axvline(0,            color=cfg.ANNO_REF,    lw=1.5, linestyle=":",  label="0s (neutral)")
ax.axvline(mean_delta,   color=cfg.ANNO_MEAN,   lw=1.5,                 label=f"Ø {mean_delta:.0f}s")
ax.axvline(median_delta, color=cfg.ANNO_MEDIAN, lw=1.5, linestyle="--", label=f"Median {median_delta:.0f}s")
ax.set_xlabel("Seconds  (Δ = departure − arrival)", **style["label"])
ax.set_ylabel("Count", **style["label"])
ax.set_title("Delay Delta — Distribution Detail · clipped −100 to +100s", **style["title"])
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Beobachtung:** Die Verteilung ist bimodal — zwei erkennbare Häufungspunkte: einer nahe 0s (neutrale Halte) und einer um −50s (systematische Recovery). Der −50s-Cluster ist wahrscheinlich kein Fehler, sondern **Terminusverhalten**: Endhalte haben oft eingebaute Pufferzeit und holen dort systematisch Zeit auf. Ohne diesen Cluster wäre der mittlere Verspätungsaufbau pro Halt noch deutlich höher als +5s. → Räumlich prüfen in `03_analysis_4-spatial`: welche Haltestellen haben systematisch delta < −30s?

## Extreme Values

Wie viele Halte liegen jenseits relevanter Schwellwerte? Gibt es echte Ausreißer oder ist die Verteilung kontinuierlich?

In [ ]:
section_header("Extreme Values")

total = lf.select(pl.len()).collect().item()
thresholds = [120, 300, 600, 1800]
rows = []
for t in thresholds:
    r = lf.select([
        (pl.col("arrival_delay")    >  t).sum().alias("arr_late"),
        (pl.col("arrival_delay")    < -t).sum().alias("arr_early"),
        (pl.col("departure_delay")  >  t).sum().alias("dep_late"),
    ]).collect()
    rows.append({
        "Threshold":  f"> {t}s  ({t//60}min)",
        "Arr Late":   f"{r['arr_late'][0]:>10,.0f}  ({r['arr_late'][0]/total:.2%})",
        "Arr Early":  f"{r['arr_early'][0]:>10,.0f}  ({r['arr_early'][0]/total:.2%})",
        "Dep Late":   f"{r['dep_late'][0]:>10,.0f}  ({r['dep_late'][0]/total:.2%})",
    })

show_df(pd.DataFrame(rows))

**Beobachtung:** Die extremsten Werte (+3000s bis +5000s) sind nicht zwingend Messfehler — bei großflächigen Störungen (Unwetter, Netzausfälle, Unfälle) können echte Kumulationsverspätungen dieser Größenordnung auftreten. Interessant wäre ein späterer Abgleich mit externen Ereignis-Daten (Wetterdaten, Störungsmeldungen) in `03_analysis_3-temporal`: Fallen die Extremwert-Häufungen zeitlich mit dokumentierten Ereignissen zusammen? Starke Frühankünfte (−200s+) konzentrieren sich vermutlich auf Terminushalte mit Pufferzeit.

## Cancellations

`canceled = True` ist der Extremfall — faktisch unendliche Verspätung. Wie viele Ausfälle gibt es insgesamt?

In [ ]:
section_header("Cancellations")

cancellations = (
    lf
    .group_by("canceled")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / pl.col("count").sum()).alias("share"))
    .sort("canceled")
    .collect()
)
log(cancellations.to_pandas().to_string())

**Beobachtung:** 3.6% klingt wenig — entspricht aber ~2.2 Mio. ausgefallenen Halt-Ereignissen im Datensatz. Mit `trip_id` ist jetzt analysierbar, ob Ausfälle einzelne Halte oder ganze Fahrten betreffen. Die Ausfallquote nach Linie folgt im nächsten Abschnitt.

### Ausfälle nach Linie

In [ ]:
section_header("Cancellations by Line")

from wgnd.core.theme import mpl_style

cancel_by_line = (
    lf
    .group_by("line_name")
    .agg([
        pl.len().alias("total"),
        pl.col("canceled").sum().alias("canceled_count"),
    ])
    .with_columns((pl.col("canceled_count") / pl.col("total")).alias("cancel_rate"))
    .sort("cancel_rate", descending=True)
    .head(15)
    .collect()
    .to_pandas()
)

style = mpl_style()
avg_rate = cancel_by_line["cancel_rate"].mean()
bar_colors = [cfg.COLOR_NEGATIVE if r > avg_rate * 1.5 else cfg.PALETTE_CATEGORICAL[4]
              for r in cancel_by_line["cancel_rate"]]

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(cancel_by_line["line_name"].astype(str), cancel_by_line["cancel_rate"], color=bar_colors)
ax.axvline(avg_rate, color=cfg.ANNO_MEAN, lw=1.5, linestyle="--", label=f"Ø {avg_rate:.1%}")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_xlabel("Cancellation Rate", **style["label"])
ax.set_title("Cancellation Rate — Top 15 Lines", **style["title"])
ax.invert_yaxis()
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
plt.tight_layout()
plt.show()

**Beobachtung:** Linie 12 sticht mit Abstand heraus — die Ausfallrate liegt rund 20× über dem Durchschnitt aller anderen Linien. Zeitliche Eingrenzung (Jahresvergleich) zeigt, dass dies auf eine **Baustellen-Phase Januar 2023 – Juni 2024** zurückzuführen ist (Streckensperrung, Ersatzverkehr). Ab Juli 2024 normalisiert sich die Rate auf ~0.2%. Für Modellierung und alle linienübergreifenden Ausfallstatistiken sollte dieser Zeitraum entweder gefiltert oder als eigenes Feature (`linie_12_baustelle`) kodiert werden. → Für spätere Analyse: Backlog-Eintrag für detaillierte Zeitraumvalidierung.

### Trip-Level Validierung

Ist `canceled` wirklich ein Trip-Level-Flag — d.h. wenn eine Fahrt ausfällt, sind **alle** Halte dieser Fahrt als `canceled = True` markiert?  
Oder gibt es "gemischte" Trips wo nur ein Teil der Halte canceled ist (→ das wären die Kurzwendungen der pre-Juli-2024-Ära)?

Gruppierung nach `trip_id` + `operating_date` — jeder Trip wird als `fully_canceled` / `fully_active` / `mixed` klassifiziert. Vergleich pre/post Juli 2024 zeigt ob sich das Muster mit der Datendefinitions-Änderung ändert.

In [ ]:
section_header("Canceled — Trip-Level Validierung")

from wgnd.core.theme import mpl_style

# trip_id ist im Master verfügbar — direkt aus Raw laden für diese Validierung
# (TRAIN/TEST Feature-Files erhalten trip_id nach dem nächsten Preparation-Run)
master_path = PATHS["raw"] / "zh-tram-data-master.parquet"

trip_cancel = (
    pl.scan_parquet(master_path)
    .select(["operating_date", "trip_id", "canceled"])
    .group_by(["trip_id", "operating_date"])
    .agg([
        pl.len().alias("n_stops"),
        pl.col("canceled").sum().alias("n_canceled"),
        pl.col("canceled").mean().alias("cancel_share"),
    ])
    .with_columns([
        pl.when(pl.col("cancel_share") == 1.0).then(pl.lit("fully_canceled"))
          .when(pl.col("cancel_share") == 0.0).then(pl.lit("fully_active"))
          .otherwise(pl.lit("mixed"))
          .alias("trip_type"),
        (pl.col("operating_date") < pl.date(2024, 7, 1)).alias("is_pre_july_2024"),
    ])
    .collect()
)

# --- Übersichtstabelle ---
summary = (
    trip_cancel
    .group_by(["is_pre_july_2024", "trip_type"])
    .agg(pl.len().alias("n_trips"))
    .sort(["is_pre_july_2024", "trip_type"])
    .with_columns(
        pl.when(pl.col("is_pre_july_2024"))
          .then(pl.lit("pre Jul 2024"))
          .otherwise(pl.lit("ab Jul 2024"))
          .alias("is_pre_july_2024")
    )
)
show_df(summary.to_pandas())

total       = len(trip_cancel)
n_mixed     = trip_cancel.filter(pl.col("trip_type") == "mixed").height
n_canceled  = trip_cancel.filter(pl.col("trip_type") == "fully_canceled").height
log(f"Gesamt Trips:             {total:>12,}")
log(f"  fully_active:           {total - n_mixed - n_canceled:>12,}  ({(total - n_mixed - n_canceled)/total:.2%})")
log(f"  fully_canceled:         {n_canceled:>12,}  ({n_canceled/total:.2%})")
log(f"  mixed (Kurzwendungen?): {n_mixed:>12,}  ({n_mixed/total:.2%})")

# --- Stacked Bar: pre vs. post Juli 2024 ---
style   = mpl_style()
colors  = {
    "fully_active":   cfg.COLOR_POSITIVE,
    "mixed":          cfg.PALETTE_CATEGORICAL[5],
    "fully_canceled": cfg.COLOR_NEGATIVE,
}
periods = ["pre Jul 2024", "ab Jul 2024"]
types   = ["fully_active", "mixed", "fully_canceled"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, normalize in zip(axes, [False, True]):
    bottoms = {p: 0 for p in periods}
    for t in types:
        vals = []
        for p in periods:
            flag = p == "pre Jul 2024"
            sub  = summary.filter(
                (pl.col("is_pre_july_2024") == p) & (pl.col("trip_type") == t)
            )
            total_p = trip_cancel.filter(pl.col("is_pre_july_2024") == flag).height
            v = sub["n_trips"][0] if len(sub) > 0 else 0
            vals.append(v / total_p if normalize else v)
        bars = ax.bar(periods, vals, bottom=[bottoms[p] for p in periods],
                      label=t, color=colors[t], alpha=0.85)
        for bar, val in zip(bars, vals):
            if val > (0.01 if normalize else 500):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_y() + bar.get_height()/2,
                        f"{val:.1%}" if normalize else f"{val:,.0f}",
                        ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        for p, v in zip(periods, vals):
            bottoms[p] += v

    ax.set_title(f"{'Anteil' if normalize else 'Anzahl'} Trips nach Typ", **style["title"])
    if normalize:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)

axes[0].legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

# --- Mixed Trips: Beispiele ---
if n_mixed > 0:
    mixed_ex = (
        trip_cancel.filter(pl.col("trip_type") == "mixed")
        .sort("n_canceled", descending=True)
        .head(5)
    )
    log("\nBeispiele 'mixed' Trips (meiste canceled Stops):")
    show_df(mixed_ex.to_pandas())

**Beobachtung:** *(Zelle noch nicht ausgeführt — benötigt trip_id in Feature-Files nach dem nächsten Preparation-Run. Erwartung: pre-Juli-2024 deutlich mehr `mixed` Trips als Nachweis für Kurzwendungs-Kodierung; ab Juli 2024 fast nur `fully_canceled` oder `fully_active`.)*

### Linie 12 — Baustelle Temporal

In [ ]:
section_header("Linie 10 & 12 — Cancellation Rate Over Time")

from wgnd.core.theme import mpl_style

cancel_monthly = (
    lf_all
    .with_columns([
        pl.col("operating_date").dt.year().alias("year"),
        pl.col("operating_date").dt.month().alias("month"),
    ])
    .group_by(["year", "month", "line_name"])
    .agg(pl.col("canceled").mean().alias("cancel_rate"))
    .sort(["year", "month"])
    .collect()
    .to_pandas()
)
cancel_monthly["date"] = pd.to_datetime(cancel_monthly[["year", "month"]].assign(day=1))

l10    = cancel_monthly[cancel_monthly["line_name"] == "10"].sort_values("date")
l12    = cancel_monthly[cancel_monthly["line_name"] == "12"].sort_values("date")
others = (cancel_monthly[~cancel_monthly["line_name"].isin(["10", "12"])]
          .groupby("date")["cancel_rate"].mean().reset_index())

style = mpl_style()

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(l12["date"],    l12["cancel_rate"],    color=line_color("12"), lw=2.5, marker="o", markersize=4, label="Linie 12")
ax.plot(l10["date"],    l10["cancel_rate"],    color=line_color("10"), lw=2.5, marker="o", markersize=4, label="Linie 10")
ax.plot(others["date"], others["cancel_rate"], color=cfg.COLOR_NEUTRAL, lw=1.5, marker="o", markersize=2, label="Ø alle anderen Linien", alpha=0.7)

ax.axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-06-30"),
           alpha=0.07, color=cfg.COLOR_NEGATIVE, label="Glattalbahn Baustelle (Jan 2023 – Jun 2024)")

for year in [2024, 2025]:
    ax.axvline(pd.Timestamp(f"{year}-01-01"), color=cfg.CHART_AXIS, lw=1, linestyle=":")

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylabel("Ausfallrate", **style["label"])
ax.set_title("Monthly Cancellation Rate — Linie 10 & 12 vs. Ø andere Linien", **style["title"])
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
plt.tight_layout()
plt.show()

log("Linie 10 & 12 — Ausfallrate nach Zeitraum:")
for linename, df in [("10", l10), ("12", l12)]:
    log(f"\n  Linie {linename}:")
    for label, mask in [
        ("2023 Jan–Dez    ", (df["date"] >= "2023-01-01") & (df["date"] <= "2023-12-31")),
        ("2024 Jan–Jun    ", (df["date"] >= "2024-01-01") & (df["date"] <= "2024-06-30")),
        ("2024 Jul–Dez    ", (df["date"] >= "2024-07-01") & (df["date"] <= "2024-12-31")),
        ("2025 Jan–Okt    ", (df["date"] >= "2025-01-01") & (df["date"] <= "2025-10-31")),
    ]:
        log(f"    {label}: {df[mask]['cancel_rate'].mean():.1%}")

**Beobachtung:** Linie 10 und Linie 12 zeigen synchrones Verhalten — aber beim Vergleich mit anderen Linien zeigt sich: **fast das gesamte Netz** hatte erhöhte Ausfallraten vor Juli 2024. Linie 9 (~10%), Linie 17 (~11%), Linie 7 (~6%) — alle normalisieren gleichzeitig im Juli 2024 auf ~0.3%, obwohl sie völlig unterschiedliche Strecken fahren.

**Hypothese: Datendefinitions-Änderung Juli 2024.** Vor diesem Datum wurden vermutlich Kurzwendungen, Teilausfälle und Betriebsanpassungen als `canceled` geführt — ab Juli 2024 nur noch echte Vollausfälle. Das erklärt die netzweite simultane Normalisierung besser als jede Baustellen-Theorie.

---

**Revidierte Strategie — `canceled`-Flag pre/post Juli 2024:**

| Strategie | Beschreibung | Pro | Con |
|:---|:---|:---|:---|
| **A — Feature kodieren** | `is_pre_july_2024 = 1` für alle Linien vor Jul 2024 | Daten bleiben, Modell bekommt Kontext | Modell muss Effekt lernen |
| **B — Zeitraum filtern** | Pre-Jul 2024 `canceled`-Records aus Training | Sauberste Baseline | Verliert ~18 Monate Daten |
| **C — canceled komplett ausschließen** | `canceled = True` Records aus Delay-Modell raus (haben keine sinnvollen Delay-Werte) | Sinnvoll — ausgefallene Fahrten haben kein `arrival_delay` | Cancellation-Modell separat behandeln |

**Entscheidung: Strategie A + C kombiniert.** `canceled = True` Records werden aus dem Delay-Modell ausgeschlossen (die haben keine sinnvollen Verspätungswerte). Für ein separates Cancellation-Modell wird `is_pre_july_2024` als Feature kodiert. → F-TARGET-05 (aktualisiert)

## Delay Delta — Monthly Trend

Wie entwickelt sich `delay_delta` über die Zeit? Gibt es Saisonalität, oder ist der Anstieg linear?

In [ ]:
section_header("Monthly Delay — All Three Years")

from wgnd.core.theme import mpl_style

monthly = (
    lf_all
    .with_columns([
        pl.col("operating_date").dt.year().alias("year"),
        pl.col("operating_date").dt.month().alias("month"),
    ])
    .group_by(["year", "month"])
    .agg([
        pl.col("arrival_delay").mean().alias("arr_mean"),
        pl.col("departure_delay").mean().alias("dep_mean"),
        pl.col("delay_delta").mean().alias("delta_mean"),
    ])
    .sort(["year", "month"])
    .collect()
    .to_pandas()
)
monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
monthly = monthly.sort_values("date")

style   = mpl_style()
colors  = cfg.palette_n(3)
metrics = [
    ("arr_mean",   "Arrival Delay",   colors[0]),
    ("dep_mean",   "Departure Delay", colors[1]),
    ("delta_mean", "Delay Delta",     colors[2]),
]

fig, ax = plt.subplots(figsize=(14, 5))
for col, label, color in metrics:
    ax.plot(monthly["date"], monthly[col], color=color, lw=2, marker="o", markersize=3, label=label)
ax.axhline(0, color=cfg.ANNO_REF, lw=1, linestyle=":")
for year in [2024, 2025]:
    ax.axvline(pd.Timestamp(f"{year}-01-01"), color=cfg.CHART_AXIS, lw=1, linestyle=":")
ax.set_ylabel("Ø Sekunden", **style["label"])
ax.set_title("Monthly Delay — Arrival · Departure · Delta — 2023–2025 (alle Monate)", **style["title"])
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
plt.tight_layout()
plt.show()

**Beobachtung:** Der monatliche Verlauf zeigt ab **November 2025** einen abrupten Sprung in `delay_delta_mean` (von ~5s auf ~17s im November, ~26s im Dezember). Dies entspricht keiner organischen Saisonschwankung — die Kurve bricht aus dem langjährigen Muster aus. Wahrscheinlichste Ursache: **Fahrplanwechsel Dezember 2025** (VBZ-Netzrestrukturierung j25→j26). Wenn neue Soll-Zeiten erst verzögert ins GTFS eingepflegt wurden, würden die Ist-Abweichungen künstlich aufgebläht erscheinen. **Nov–Dez 2025 aus Trendanalysen ausschließen.** → Bereinigte Ansicht folgt direkt unten.

In [ ]:
section_header("Monthly Delay — Jan 2023 – Okt 2025 (bereinigt + Trend)")

from wgnd.core.theme import mpl_style

# Nov + Dez 2025 raus — Fahrplanwechsel-Artefakt
monthly_clean = monthly[
    ~((monthly["year"] == 2025) & (monthly["month"] >= 11))
].copy().reset_index(drop=True)

style   = mpl_style()
colors  = cfg.palette_n(3)
metrics = [
    ("arr_mean",   "Arrival Delay",   colors[0]),
    ("dep_mean",   "Departure Delay", colors[1]),
    ("delta_mean", "Delay Delta",     colors[2]),
]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(monthly_clean))

for col, label, color in metrics:
    y = monthly_clean[col].values
    ax.plot(monthly_clean["date"], y, color=color, lw=2, marker="o", markersize=3, label=label)
    # Lineare Trendlinie
    coeffs = np.polyfit(x, y, 1)
    trend  = np.polyval(coeffs, x)
    ax.plot(monthly_clean["date"], trend, color=color, lw=1.5, linestyle="--", alpha=0.6)

ax.axhline(0, color=cfg.ANNO_REF, lw=1, linestyle=":")
for year in [2024, 2025]:
    ax.axvline(pd.Timestamp(f"{year}-01-01"), color=cfg.CHART_AXIS, lw=1, linestyle=":")
ax.set_ylabel("Ø Sekunden", **style["label"])
ax.set_title("Monthly Delay — Jan 2023 – Okt 2025 · gestrichelt = linearer Trend", **style["title"])
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
plt.tight_layout()
plt.show()

**Beobachtung:** Ohne den Fahrplanwechsel-Artefakt zeigt sich ein klares saisonales Muster: **Winter-Peak (Dez/Jan)** und ein kleinerer **Frühlings-Peak (März)** sowie **Sommer-Peak (Juni)** — unterbrochen von einem relativen Tal in den Sommermonaten (Juli–August), das aber trotzdem auf hohem Niveau bleibt. Die gestrichelten Trendlinien bestätigen einen **strukturellen Aufwärtstrend** über alle drei Metriken — kein Einmaleffekt. `dep_delay` steigt am stärksten. Alle drei Metriken steigen: das System wird insgesamt langsamer, nicht nur an einzelnen Punkten. → Saison-Feature (Monat, Winter/Sommer-Flag) und Jahr als Features in Modell aufnehmen.

### Hintergrund: VBZ Fahrplanwechsel 14. Dezember 2025

Recherchierte Quellen bestätigen: Am **14. Dezember 2025** trat der grösste Fahrplanwechsel in der Geschichte der VBZ in Kraft — **"Tramnetz Süd"**. 7 von 14 Tramlinien fahren seither auf neuen Strecken. Die Haltestelle Bahnhofquai/HB wurde für ein Jahr gesperrt, zwei neue Baustellenlinien (50 + 51) eingeführt.

**Warum bereits Oktober/November?** GTFS-S Daten werden wöchentlich (donnerstags) publiziert. Die j26-GTFS-Dateien wurden in der Vorbereitungsphase (ca. Nov 2025) ins Datensystem eingespielt — bevor die physischen Linien tatsächlich umgestellt waren. Ein IST-Record aus November, der gegen j26-Soll-Zeiten auf veränderten Strecken abgeglichen wird, erzeugt künstliche Verspätungen.

**j25 vs. j26:** j25 = Jahresfahrplan 2025, gültig Dez 2024 – 13. Dez 2025. j26 = Jahresfahrplan 2026, gültig ab 14. Dez 2025. Beide GTFS-Dateien sind öffentlich verfügbar auf [opentransportdata.swiss](https://data.opentransportdata.swiss/dataset/timetable-2025-gtfs2020).

Quellen: [VBZ Fahrplanwechsel](https://fahrplanwechsel.vbz.ch/) · [Stadt Zürich Medienmitteilung](https://www.stadt-zuerich.ch/vbz/de/die-vbz/medien/medienmitteilungen/2025/september/groesster-fahrplanwechsel-in-der-geschichte-der-vbz.html) · [ZVV Medienmitteilung 26.11.2025](https://www.zvv.ch/de/ueber-uns/zuercher-verkehrsverbund/medien/medienmitteilungen/2025/2025-11-26-ZVV-Fahrplanwechsel-bringt-grosse-Aenderungen.html) · [NZZ Übersicht](https://www.nzz.ch/visuals/fahrplanwechsel-2025-zuerichs-tramlinien-fahren-auf-neuen-wegen-ld.1908278)

---

**Strategien für den Umgang mit dem Artefakt:**

| Strategie | Beschreibung | Aufwand | Empfehlung |
|:---|:---|:---|:---|
| **A — Entfernen** | Nov–Dez 2025 aus Train + Test raus | Minimal | ✅ Jetzt — sauberste Lösung |
| **B — Binary-Feature** | `fahrplanwechsel = 1` ab 14.12.2025 | Gering | Möglich, kausaler Zusammenhang aber unklar |
| **C — Re-Annotation** | IST-Daten Nov–Dez 2025 mit j25-GTFS neu berechnen | Hoch — benötigt `trip_id` + j25-GTFS-Join | Später wenn `trip_id` in Pipeline |
| **D — Separate Evaluation** | Modell auf Nov–Dez 2025 separat evaluieren (Out-of-distribution) | Mittel | Ergänzend zu A |

**Aktuell gewählt: Strategie A.** Nov–Dez 2025 werden aus allen Trendanalysen ausgeschlossen. Train + Test gilt: Jan 2023 – Okt 2025. Strategie C als Option dokumentiert — Voraussetzung: `trip_id` in Pipeline (→ F-TARGET-08).

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Handlungsempfehlungen in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-TARGET-01 | `arrival_delay` rechtsschiefe Verteilung — Log-Transform empfohlen | open |
| F-TARGET-02 | `delay_delta` bimodal — Terminus-Cluster bei −50s | open |
| F-TARGET-03 | 70% `delay_delta > 0` — kein ausreichender Fahrplanpuffer | open |
| F-TARGET-04 | Scheduled Dwell-Time (`dep_schedule − arr_schedule`) als Puffer-Feature verfügbar | open |
| F-TARGET-05 | `canceled`-Flag netzweit erhöht Jan 2023 – Jun 2024 — wahrscheinlich Datendefinitions-Änderung | open |
| F-TARGET-06 | Nov–Dez 2025 Fahrplanwechsel-Artefakt — aus Analysen ausgeschlossen (Strategie A) | ⚠️ aktiv |
| F-TARGET-07 | Extremwerte bis +5000s — Abgleich mit Ereignis-Daten ausstehend | open |
| F-TARGET-08 | `trip_id` und `stop_sequence` jetzt im Master-Datensatz (26 Spalten) — Kaskaden- und Trip-Level-Analyse möglich | done |
| F-TARGET-09 | Langfristiger Delta-Aufwärtstrend +4.4s (2023) → +7.7s (2025) | open |